### Name: Blessing Adeniji
### Degree: MSc Artifical Intelligence Online
### Capstone Project: AI-Generated Text Detection - Deepfakes
Purpose: 100-text sample evaluation
25 texts from each of the four test sets, balanced Human/AI, run through all six MAGE-trained models for a direct comparison.
Small enough to inspect individaual cases and the right size for a Pangram API comparison if access becomes available

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

SEED = 42 # fixed number so the sample is reproducible

# The four test sets
test_files = {
    "MAGE":     "data_splits/MAGE_test.csv",
    "ChatGPT-Research-Abstracts": "data_splits/ChatGPT-Research-Abstracts_test.csv",
    "GPT-Wiki-Intro":   "data_splits/GPT-Wiki-Intro_test.csv",
    "RAID":             "data_splits/RAID_test.csv"
}

samples = []
for dataset_name, path in test_files.items():
    df = pd.read_csv(path)
    # 13 human (label 0) and 12 AI (label 1) = 25 per dataset
    human = df[df["label"] == 0].sample(n=13, random_state=SEED)
    ai    = df[df["label"] == 1].sample(n=12, random_state=SEED)
    part  = pd.concat([human, ai])[["text", "label"]]
    part["dataset"] = dataset_name
    samples.append(part)

# Combine and shuffle
sample_df = pd.concat(samples).sample(frac=1, random_state=SEED).reset_index(drop=True)

os.makedirs("sample_evaluation", exist_ok=True)
sample_df.to_csv("sample_evaluation/100_samples_texts.csv", index=False)

# Print results
print("Total texts:", len(sample_df))
print("\nPer dataset:")
print(sample_df.groupby(["dataset", "label"]).size())

Total texts: 100
/nPer dataset:
dataset                     label
ChatGPT-Research-Abstracts  0        13
                            1        12
GPT-Wiki-Intro              0        13
                            1        12
MAGE                        0        13
                            1        12
RAID                        0        13
                            1        12
dtype: int64


In [4]:
# Evaluate all six MAGE-trained models on the 100-text sample.
# Each model predicts every text; predictions are saved per text so
# individual cases and disagreements can be inspected.

sample_df = pd.read_csv("sample_evaluation/100_samples_texts.csv")

models = {
    "ettin-encoder-68m":  "models/ettin68m_mage_final",
    "ettin-decoder-68m":  "models/decoder_mage_final",
    "roberta-base":       "models/roberta_mage_final",
    "modernbert-base":    "models/modernbert_mage_final",
    "ettin-encoder-400m": "models/ettin_encoder400m_mage_final",
    "ettin-decoder-400m": "models/ettin_decoder400m_mage_final",
}

results_table = sample_df[["dataset", "label", "text"]].copy()

for model_name, path in models.items():
    tokenizer = AutoTokenizer.from_pretrained(path)
    if tokenizer.pad_token is None:                      # decoders need this
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(path)
    model.config.pad_token_id = tokenizer.pad_token_id

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)
    ds = Dataset.from_pandas(sample_df[["text", "label"]]).map(tokenize, batched=True)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="models/tmp_predict", per_device_eval_batch_size=8, report_to="none"),
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    preds = np.argmax(trainer.predict(ds).predictions, axis=1)
    results_table[model_name] = preds

    acc = (preds == sample_df["label"]).mean()
    print(f"{model_name}: {acc:.1%}")

# Accuracy per model, broken down by source dataset
print("\nAccuracy by dataset:")
summary = {}
for model_name in models:
    correct = results_table[model_name] == results_table["label"]
    summary[model_name] = correct.groupby(results_table["dataset"]).mean()
summary_df = pd.DataFrame(summary).T
summary_df["OVERALL"] = [(results_table[m] == results_table["label"]).mean() for m in models]
print((summary_df * 100).round(1))

# Save both the per-text detail and the summary
results_table.to_csv("sample_evaluation/100_samples_text_predictions.csv", index=False)
summary_df.to_csv("sample_evaluation/100_samples_text_summary.csv")

# Add an empty column ready for Pangram, if access becomes available
results_table["pangram_pred"] = ""
results_table.to_csv("sample_evaluation/100_samples_text_for_pangram.csv", index=False)
print("\nSaved. Pangram-ready file: sample_evaluation/100_samples_text_for_pangram.csv")

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

ettin-encoder-68m: 86.0%


Loading weights:   0%|          | 0/157 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

ettin-encoder-400m: 88.0%


Loading weights:   0%|          | 0/229 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

ettin-decoder-400m: 93.0%

Accuracy by dataset:
dataset             ChatGPT-Research-Abstracts  GPT-Wiki-Intro  MAGE  RAID  \
ettin-encoder-68m                         80.0            96.0  92.0  76.0   
ettin-decoder-68m                         88.0            96.0  84.0  84.0   
roberta-base                              92.0            80.0  84.0  80.0   
modernbert-base                           84.0            92.0  92.0  72.0   
ettin-encoder-400m                        92.0            92.0  92.0  76.0   
ettin-decoder-400m                        92.0            96.0  96.0  88.0   

dataset             OVERALL  
ettin-encoder-68m      86.0  
ettin-decoder-68m      88.0  
roberta-base           84.0  
modernbert-base        85.0  
ettin-encoder-400m     88.0  
ettin-decoder-400m     93.0  

Saved. Pangram-ready file: sample_evaluation/100_samples_text_for_pangram.csv


In [5]:
# Which texts did most models get wrong? These are the interesting cases.
model_cols = list(models.keys())
results_table["n_wrong"] = sum((results_table[m] != results_table["label"]).astype(int) for m in model_cols)
hard = results_table[results_table["n_wrong"] >= 4].sort_values("n_wrong", ascending=False)
print(f"Texts that at least 4 of 6 models got wrong: {len(hard)}\n")
for _, row in hard.head(5).iterrows():
    truth = "HUMAN" if row["label"] == 0 else "AI"
    print(f"[{row['dataset']}] true={truth}, {row['n_wrong']}/6 models wrong")
    print(row["text"][:300], "...\n")

Texts that at least 4 of 6 models got wrong: 8

[RAID] true=AI, 6/6 models wrong
Ian Austin (born 25 April 1978) is a British former politician. He was first elected Member of Parliament for Dudley North in 2005, defeating the Conservative candidate Stuart Turner. He was re-elected in 2010 with an increased majority. He served as a Conservative MP until he resigned after the 201 ...

[MAGE] true=AI, 6/6 models wrong
Mike was loosing his hair. He was self conscious about his hair loss. Mike looked online for toupees. Mike ordered a toupee online. The toupee arrived two weeks later. Mike felt regret he didn't buy earlier. ...

[RAID] true=HUMAN, 6/6 models wrong
M​e​d​i​c​a​l​ ​i​m​a​g​e​ ​s​e​g​m​e​n​t​a​t​i​o​n​ ​r​e​q​u​i​r​e​s​ ​c​o​n​s​e​n​s​u​s​ ​g​r​o​u​n​d​ ​t​r​u​t​h​ ​s​e​g​m​e​n​t​a​t​i​o​n​s​ ​t​o​
​b​e​ ​d​e​r​i​v​e​d​ ​f​r​o​m​ ​m​u​l​t​i​p​l​e​ ​e​x​p​e​r​t​ ​a​n​n​o​t​a​t​i​o​n​s​.​ ​A​ ​n​o​v​e​l​ ​a​p​p​r​o​a​c​h​ ​i​s​ ​p​r​o​p​o​s​e​d​ ​ ...

[ChatGPT-Research-Abstrac

In [6]:
hard.to_csv("sample_evaluation/hard_cases.csv", index=False)
print(f"Saved {len(hard)} hard cases for the write-up.")

Saved 8 hard cases for the write-up.
